In [ ]:
# GNN generation + identification of samples with extreme base-pairing probabilities
# Training/model/Optuna/data-splitting code has been removed.

import csv
import numpy as np
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)

########################################
# 1. Input files and thresholds
########################################

GUIDE_FILE = "RfxCas13d_validation_full_guide_sequences_combined.txt"
TARGET_FILE = "RfxCas13d_validation_target_sequences_noPAM_combined.txt"

# The original GNN code only creates an edge when pairing probability > 0.01.
EDGE_THRESHOLD = 0.01

# Every relevant generated base-pair edge must be "extreme":
# probability > 0.98 OR probability < 0.02.
HIGH_PROB_THRESHOLD = 0.98
LOW_PROB_THRESHOLD = 0.02

# Last 30 nt of the full guide are the spacer.
SPACER_LENGTH = 30

# Output files
UNION_INDEX_FILE = "outlier_feature_index_GNN.txt"
DETAIL_CSV_FILE = "extreme_probability_sample_details.csv"

########################################
# 2. Original GNN helper functions
########################################

def nucleotide_to_one_hot(nucleotide):
    mapping = {
        "A": [1, 0, 0, 0],
        "C": [0, 1, 0, 0],
        "G": [0, 0, 1, 0],
        "T": [0, 0, 0, 1],
        "U": [0, 0, 0, 1],
        "-": [0, 0, 0, 0],
    }
    return mapping.get(nucleotide, [0, 0, 0, 0])


def encode_sequence(sequence):
    return [nucleotide_to_one_hot(nuc) for nuc in sequence]


def matrix_to_edge_index(prob_matrix, threshold=EDGE_THRESHOLD):
    """
    Same graph-edge rule as the original notebook:
    keep only i < j edges whose pairing probability is > 0.01.
    """
    edges_source = []
    edges_target = []
    L = len(prob_matrix)

    for i in range(L):
        for j in range(i + 1, L):
            if prob_matrix[i][j] > threshold:
                edges_source.append(i)
                edges_target.append(j)

    return [edges_source, edges_target]


def generate_edge_features(edge_index, seq_one_hot, prob_matrix):
    """
    Preserve the original edge features:
        [pair_type / 3, pairing_probability]
    """
    features = []

    if len(edge_index[0]) == 0:
        return features

    def decode_nt(one_hot_vec):
        if one_hot_vec[0] == 1:
            return "A"
        elif one_hot_vec[1] == 1:
            return "C"
        elif one_hot_vec[2] == 1:
            return "G"
        elif one_hot_vec[3] == 1:
            return "T"
        return "N"

    for src_idx, tgt_idx in zip(edge_index[0], edge_index[1]):
        prob_val = float(prob_matrix[src_idx][tgt_idx])

        src_char = decode_nt(seq_one_hot[src_idx])
        tgt_char = decode_nt(seq_one_hot[tgt_idx])

        pair_type = 0

        if (
            (src_char == "A" and tgt_char in ["T", "U"])
            or (src_char in ["T", "U"] and tgt_char == "A")
        ):
            pair_type = 2
        elif (
            (src_char == "G" and tgt_char in ["T", "U"])
            or (src_char in ["T", "U"] and tgt_char == "G")
        ):
            pair_type = 2
        elif (
            (src_char == "C" and tgt_char == "G")
            or (src_char == "G" and tgt_char == "C")
        ):
            pair_type = 3

        features.append([pair_type / 3, prob_val])

    return features


def RNA_reverse_complement(RNA):
    complement = {'A': 'U', 'C': 'G', 'G': 'C', 'U': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(RNA))


########################################
# 3. NUPACK models
########################################

# These are kept identical to the original notebook.
my_model_RNA = Model(
    material="rna",
    ensemble="nostacking",
    celsius=37,
    sodium=0.1,
    magnesium=0.01,
)

my_model_DNA = Model(
    material="dna",
    ensemble="nostacking",
    celsius=37,
    sodium=0.1,
    magnesium=0.01,
)


########################################
# 4. Extreme-probability checking
########################################

def is_extreme_probability(p):
    """
    Strictly follows the requested cutoffs:
        p > 0.90 OR p < 0.10

    Because the original graph only contains edges with p > 0.01,
    the low-probability graph edges considered here are:
        0.01 < p < 0.10.
    """
    return (p > HIGH_PROB_THRESHOLD) or (p < LOW_PROB_THRESHOLD)


def get_graph_edge_records(prob_matrix, edge_index):
    """
    Return all generated GNN edges as:
        (source_index, target_index, probability)
    """
    records = []

    for src, tgt in zip(edge_index[0], edge_index[1]):
        records.append((src, tgt, float(prob_matrix[src][tgt])))

    return records


def extreme_edges(edge_records):
    """Return generated graph edges with probability >0.98 or <0.02."""
    return [
        (src, tgt, p)
        for src, tgt, p in edge_records
        if is_extreme_probability(p)
    ]


def all_edges_are_extreme(edge_records):
    """
    True only when ALL relevant generated base-pair edges satisfy:
        p > 0.98 OR p < 0.02

    Empty edge sets are NOT considered to satisfy the criterion.
    """
    return (
        len(edge_records) > 0
        and all(is_extreme_probability(p) for _, _, p in edge_records)
    )


def format_edges(records):
    """
    Compact text representation for the CSV.
    Nucleotide/node indices are 0-based.
    """
    return "; ".join(
        f"{src}-{tgt}:{p:.6f}"
        for src, tgt, p in records
    )


########################################
# 5. Generate all four GNNs and record sample indices
########################################

def generate_gnns_and_find_extreme_samples():
    with open(GUIDE_FILE, "r") as f:
        gRNA_sequences = [line.strip() for line in f if line.strip()]

    with open(TARGET_FILE, "r") as f:
        target_sequences = [line.strip() for line in f if line.strip()]

    if len(gRNA_sequences) != len(target_sequences):
        raise ValueError(
            f"Guide/target sample counts differ: "
            f"{len(gRNA_sequences)} guides vs {len(target_sequences)} targets."
        )

    n_samples = len(gRNA_sequences)
    print(f"Number of samples: {n_samples}")

    # Preserve all four graph-generation outputs.
    gRNA_node_features = []
    target_bh_node_features = []
    duplex_node_features = []

    gRNA_edge_indices = []
    target_bh_edge_indices = []
    duplex_edge_indices = []

    gRNA_edge_features = []
    target_bh_edge_features = []
    duplex_edge_features = []

    # Per-condition sample indices.
    criterion_1_indices = []
    criterion_2_indices = []
    criterion_3_indices = []

    details = []

    for sample_idx, (guide, target) in enumerate(
        zip(gRNA_sequences, target_sequences)
    ):

        target_bh_sequence = target
        duplex_sequence = guide + target

        # --------------------------------
        # 1) gRNA graph
        # --------------------------------
        gRNA_oh = encode_sequence(guide)
        gRNA_node_features.append(gRNA_oh)

        gRNA_prob = pairs(
            strands=guide,
            model=my_model_RNA,
        ).to_array()

        gRNA_edges = matrix_to_edge_index(gRNA_prob)
        gRNA_feats = generate_edge_features(
            gRNA_edges,
            gRNA_oh,
            gRNA_prob,
        )

        gRNA_edge_indices.append(gRNA_edges)
        gRNA_edge_features.append(gRNA_feats)

        # Restrict criterion 1 to base pairs entirely within
        # the LAST 30 nt of the full guide.
        spacer_start = max(0, len(guide) - SPACER_LENGTH)

        gRNA_spacer_records = [
            (src, tgt, float(gRNA_prob[src][tgt]))
            for src, tgt in zip(gRNA_edges[0], gRNA_edges[1])
            if src >= spacer_start and tgt >= spacer_start
        ]

        gRNA_extreme = extreme_edges(gRNA_spacer_records)
        criterion_1 = all_edges_are_extreme(gRNA_spacer_records)

        # --------------------------------
        # 2) target_bh graph
        # --------------------------------
        target_bh_oh = encode_sequence(target_bh_sequence)
        target_bh_node_features.append(target_bh_oh)

        target_bh_prob = pairs(
            strands=target,
            model=my_model_RNA,
        ).to_array()

        target_bh_edges = matrix_to_edge_index(target_bh_prob)
        target_bh_feats = generate_edge_features(
            target_bh_edges,
            target_bh_oh,
            target_bh_prob,
        )

        target_bh_edge_indices.append(target_bh_edges)
        target_bh_edge_features.append(target_bh_feats)

        target_bh_records = get_graph_edge_records(
            target_bh_prob,
            target_bh_edges,
        )
        target_bh_extreme = extreme_edges(target_bh_records)
        criterion_2 = all_edges_are_extreme(target_bh_records)

        # --------------------------------
        # 3) duplex graph
        # --------------------------------
        duplex_oh = encode_sequence(duplex_sequence)
        duplex_node_features.append(duplex_oh)

        duplex_prob = pairs(
            strands=[guide, target],
            model=my_model_RNA,
        ).to_array()

        duplex_edges = matrix_to_edge_index(duplex_prob)
        duplex_feats = generate_edge_features(
            duplex_edges,
            duplex_oh,
            duplex_prob,
        )

        duplex_edge_indices.append(duplex_edges)
        duplex_edge_features.append(duplex_feats)

        duplex_records = get_graph_edge_records(
            duplex_prob,
            duplex_edges,
        )
        duplex_extreme = extreme_edges(duplex_records)
        criterion_3 = all_edges_are_extreme(duplex_records)

        # --------------------------------
        # Record criteria
        # --------------------------------
        if criterion_1:
            criterion_1_indices.append(sample_idx)
        if criterion_2:
            criterion_2_indices.append(sample_idx)
        if criterion_3:
            criterion_3_indices.append(sample_idx)

        selected = (
            criterion_1
            or criterion_2
            or criterion_3
        )

        details.append({
            "sample_index_0based": sample_idx,
            "sample_index_1based": sample_idx + 1,
            "criterion_1_gRNA_spacer": criterion_1,
            "criterion_2_target_bh": criterion_2,
            "criterion_3_duplex": criterion_3,
            "selected_by_any_criterion": selected,
            "gRNA_spacer_extreme_edges": format_edges(gRNA_extreme),
            "target_bh_extreme_edges": format_edges(target_bh_extreme),
            "duplex_extreme_edges": format_edges(duplex_extreme)
        })

    # Union of the four criteria.
    all_extreme_indices = sorted(
        set(criterion_1_indices)
        | set(criterion_2_indices)
        | set(criterion_3_indices)
    )

    ########################################
    # 6. Save outputs
    ########################################

    # Main requested file: 0-based sample indices, one per line.
    with open(UNION_INDEX_FILE, "w") as f:
        for idx in all_extreme_indices:
            f.write(f"{idx}\n")

    # Detailed table showing exactly which criterion selected each sample.
    fieldnames = [
        "sample_index_0based",
        "sample_index_1based",
        "criterion_1_gRNA_spacer",
        "criterion_2_target_bh",
        "criterion_3_duplex",
        "selected_by_any_criterion",
        "gRNA_spacer_extreme_edges",
        "target_bh_extreme_edges",
        "duplex_extreme_edges"
    ]

    with open(DETAIL_CSV_FILE, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(details)

    print("\nFinished.")
    print(f"Criterion 1 (gRNA spacer): {len(criterion_1_indices)} samples")
    print(f"Criterion 2 (target_bh):   {len(criterion_2_indices)} samples")
    print(f"Criterion 3 (duplex):      {len(criterion_3_indices)} samples")
    print(f"Union (any criterion):     {len(all_extreme_indices)} samples")
    print()
    print(f"Main index file: {UNION_INDEX_FILE}")
    print(f"Detailed CSV:    {DETAIL_CSV_FILE}")

    return {
        "gRNA_node_features": gRNA_node_features,
        "target_bh_node_features": target_bh_node_features,
        "duplex_node_features": duplex_node_features,
        "gRNA_edge_indices": gRNA_edge_indices,
        "target_bh_edge_indices": target_bh_edge_indices,
        "duplex_edge_indices": duplex_edge_indices,
        "gRNA_edge_features": gRNA_edge_features,
        "target_bh_edge_features": target_bh_edge_features,
        "duplex_edge_features": duplex_edge_features,
        "criterion_1_indices": criterion_1_indices,
        "criterion_2_indices": criterion_2_indices,
        "criterion_3_indices": criterion_3_indices,
        "all_extreme_indices": all_extreme_indices,
        "details": details,
    }


########################################
# 7. Run
########################################

results = generate_gnns_and_find_extreme_samples()

# The final requested union of sample indices is also available in memory:
extreme_sample_indices = results["all_extreme_indices"]

print("\n0-based indices selected by ANY criterion:")
print(extreme_sample_indices)
